# Getting Started with the zahner_link Python Package

This tutorial serves as your first step in learning how to use the [zahner_link](https://doc.zahner.de/im7/apis/zahner_link/python/) Python library to control Zahner IM7 series potentiostats. We'll walk through the basic operations needed to perform simple electrochemical measurements.

This notebook covers the essential workflow:
*   **Connecting to your device:** How to establish a network connection between Python and your IM7 potentiostat
*   **Initial calibration:** Running the necessary DC calibration after warm-up
*   **Controlling the potentiostat:** Switching on the potentiostat and setting up basic parameters
*   **Running a measurement:** Performing a simple polarization test and collecting the data
*   **Saving your results:** Exporting measurement data to Zahner's XML format for further analysis
*   **Proper shutdown:** Correctly switching off the potentiostat and closing the connection

Understanding these basics will prepare you for the more advanced features and complex measurement sequences available in the [zahner_link](https://doc.zahner.de/im7/apis/zahner_link/python/) library.

Error handling is ignored in this example; this will be covered in a separate example.

In [1]:
import zahner_link as zl

## Connecting to the IM7

Ensure your IM7 device is powered on and fully booted before connecting.

To establish a connection, create a [ZahnerLinkExc](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc) object using the IM7's IP address and port number.

The [connect()](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc.connect) method returns a [ErrorObject](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ErrorObject) indicating whether the connection was successful, or it raises an exception containing the error status if the exception library is used.

As an alternative, there is also the non-exception-based version of the [ZahnerLink](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLink) class.

If the error code is [ErrorCodeEnum.PROTOCOL_VERSION_MISMATCH](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ErrorCodeEnum.PROTOCOL_VERSION_MISMATCH), you must update the IM7 firmware using Zahner Lab to proceed.

In [2]:
link = zl.ZahnerLinkExc("10.10.253.154", "1994")
error: zl.ErrorObject = link.connect()

if not error:
    print("connected successfully")
else:
    print(f"failed to connect, status: {error.get_error_code_enum()}, message: {error.get_message_formatted()}")

connected successfully


## Performing DC Calibration

After your IM7 has been warming up for **at least 30 minutes**, you should perform a [DC calibration](https://doc.zahner.de/im7/faq/index.html#warm-up-dc-calibration). This process ensures accurate measurements and takes several minutes to complete.

We create a [CalibrateJob](https://doc.zahner.de/im7/apis/zahner_link/python/pages/calibration.html#zahner_link.calibration.CalibrateJob) object and execute it using the [do_job()](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc.do_job) method.

In [ ]:
dc_calibration_job = zl.calibration.CalibrateJob()
link.do_job(dc_calibration_job)

## Switching On the Potentiostat

Before taking any measurements, we need to turn on a potentiostat using the [SwitchOnJob](https://doc.zahner.de/im7/apis/zahner_link/python/pages/control.html#zahner_link.control.SwitchOnJob).

In the code below, we configure several important parameters:
- `potentiostat`: Specifies which potentiostat to use (*"MAIN:1:POT"* is the standard main potentiostat)
- `coupling`: Sets whether we're controlling voltage ([POTENTIOSTATIC](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.PotentiostatCoupling.POTENTIOSTATIC)) or current ([GALVANOSTATIC](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.PotentiostatCoupling.GALVANOSTATIC))
- `bias`: The dc setpoint - 1.0 V in this example
- `voltage_range_index` and `compliance_range_index`: Measurement range settings for the specific device

**All values use SI base units (volts, amperes, seconds), not scaled units like mV or µA.**

The [zahner_link](https://doc.zahner.de/im7/apis/zahner_link/python/) library provides detailed documentation for each method and parameter. You can access comprehensive descriptions of all parameters, return values, and error conditions through the context help system or through the [official documentation](https://doc.zahner.de/im7/apis/zahner_link/python/). This makes it easy to understand exactly what each parameter does and how to use the full capabilities of your potentiostat.

In [ ]:
switch_on_job = zl.control.SwitchOnJob(
    potentiostat="MAIN:1:POT",
    coupling=zl.PotentiostatCoupling.POTENTIOSTATIC,
    bias=1.0,  # 1 V
    voltage_range_index=0,
    compliance_range_index=0,
)
link.do_job(switch_on_job)

## Running a Measurement

Now we can perform a polarization measurement using [PogaJob](https://doc.zahner.de/im7/apis/zahner_link/python/pages/meas.html#zahner_link.meas.PogaJob) (POtentiostatic/GAlvanostatic job).

This example configures a 5-second potentiostatic polarization at 1.0 V with these parameters:

- `bias`: The dc setpoint (1 V in potentiostatic mode)
- `time`: Duration of the measurement (5 seconds)
- `output_data_rate`: How many data points per second to record (25 Hz)
- `autorange`: When `True`, the system automatically selects the best measurement range
- `current_range`: Only used when autorange is `False`

**Important:** If you run the same job object multiple times, the previous measurement data will be overwritten.

In [ ]:
potentiostatic_polarization_job = zl.meas.PogaJob(
    bias=1,  # 1 V
    duration=5.0,
    output_data_rate=25,
    autorange=True,
    current_range=0.1,
)
link.do_job(potentiostatic_polarization_job)

## Saving Measurement Data

After collecting data, you'll typically want to save it for further analysis. The code below shows how to save measurement data in Zahner's XML format, which can be opened directly in [Zahner Analysis software](https://zahner.de/de/products-details/software/zahner-analysis).

We first acquire the measurement data from the device with [get_job_result_data()](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc.get_job_result_data) and create a [Measurement](https://doc.zahner.de/im7/apis/zahner_link/python/pages/xml.html#zahner_link.xml.Measurement) object from our job's data, then use [ZXmlExporter](https://doc.zahner.de/im7/apis/zahner_link/python/pages/xml.html#zahner_link.xml.ZXmlExporter) to save it to a file:

In [6]:
measurement_data = link.get_job_result_data(potentiostatic_polarization_job)
xml_measurement = zl.xml.Measurement(measurement_data)

exporter = zl.xml.ZXmlExporter()
exporter.set_compact_xml(False)
exporter.save_as_file_standalone(xml_measurement, "polarization.zmx")

0

## Switching Off the Potentiostat

When you're finished with your measurements, you should always switch off the potentiostat using [SwitchOffJob](https://doc.zahner.de/im7/apis/zahner_link/python/pages/control.html#zahner_link.control.SwitchOffJob).

This code switches off the specific potentiostat we've been using. Alternatively, you could use [SwitchAllOffJob](https://doc.zahner.de/im7/apis/zahner_link/python/pages/control.html#zahner_link.control.SwitchAllOffJob) to switch off all potentiostats in the system at once.

In [ ]:
switch_off_job = zl.control.SwitchOffJob(potentiostat="MAIN:1:POT")
link.do_job(switch_off_job)

## Disconnecting from the IM7

As the final step, we [disconnect()](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc.disconnect) our [ZahnerLinkExc](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc) object from the IM7 device.

After disconnecting, you can no longer execute new jobs, but you can still access any data that was already collected by previous jobs.

In [8]:
link.disconnect()